In [2]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

In [3]:
with open("knowledge_base.json", "r") as f:
    kb = json.load(f)

print(f"Loaded: {len(kb)} articles")

Loaded: 10 articles


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

for doc in kb:
    doc["vector"] = model.encode(doc["text"])

print("All articles have been embedded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All articles have been embedded!


In [5]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
def search(query, top_k=3):
    query_vector = model.encode(query)
    
    scores = []
    for doc in kb:
        score = cosine_similarity(query_vector, doc["vector"])
        scores.append((score, doc))
    
    scores.sort(key=lambda x: x[0], reverse=True)
    return scores[:top_k]

In [7]:
queries = [
    "my laptop won't switch on",
    "how do I stop being billed every month?",
    "access denied error when saving a file",
    "where do I leave my car in the evening?"
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 60)
    results = search(query, top_k=3)
    for i, (score, doc) in enumerate(results, 1):
        print(f"{i}. [{score:.2f}] ({doc['id']}) {doc['text'][:80]}...")


Query: 'my laptop won't switch on'
------------------------------------------------------------
1. [0.44] (kb-02) To power up a device that won't turn on, hold the power button for ten seconds, ...
2. [0.18] (kb-07) Reset your password from the login screen by clicking 'Forgot password'. A reset...
3. [0.16] (kb-09) Company laptops back up automatically to the cloud every night at 2am while conn...

Query: 'how do I stop being billed every month?'
------------------------------------------------------------
1. [0.54] (kb-05) To cancel your subscription, open Account Settings and choose End Plan. Cancella...
2. [0.30] (kb-06) Premium plan members get priority support, with a guaranteed first response with...
3. [0.21] (kb-10) The office kitchen is restocked every Monday and Thursday. Please label personal...

Query: 'access denied error when saving a file'
------------------------------------------------------------
1. [0.54] (kb-08) The error code 0x80070005 means 'access denied'. Run

## Reflection

For each query, the top result was the correct passage — even though none of the queries 
shared meaningful words with their best match:

| Query | Best match | Shared words? |
|---|---|---|
| "my laptop won't switch on" | kb-02 "power up a device that won't turn on" | "won't" only |
| "how do I stop being billed every month?" | kb-05 "cancel your subscription" | None |
| "access denied error when saving a file" | kb-08 "error code 0x80070005 means access denied" | "access denied", "error" |
| "where do I leave my car in the evening?" | kb-01 "park in lot B after 6pm" | None |

**What this shows:**
The embedding model captured meaning, not spelling. "Switch on" and "power up" are 
different words but landed close in vector space. "Stop being billed" and 
"cancel your subscription" share zero words yet scored 0.54 — the highest match.

**On similarity scores:**
The scores varied across queries (0.33 – 0.54). The lowest score was for the parking 
query (0.33), likely because "car" and "evening" are more indirect translations of 
"lot B" and "6pm". Despite the lower score, the correct passage still ranked first.

**Key takeaway:**
Retrieval is not keyword matching — it is comparing directions in meaning-space. 
The right passage surfaced every time using nothing but a dot product.

In [9]:
# Stretch: query not covered by the knowledge base
result = search("what's the wifi password?", top_k=1)
score, doc = result[0]
print(f"Top score: {score:.2f}")
print(f"Best match: ({doc['id']}) {doc['text'][:80]}...")

if score < 0.4:
    print("\nBelow threshold — this question is not covered in the knowledge base.")
else:
    print("\nAbove threshold — a match was found.")

Top score: 0.32
Best match: (kb-07) Reset your password from the login screen by clicking 'Forgot password'. A reset...

Below threshold — this question is not covered in the knowledge base.


## Stretch: Out-of-scope query

Query: "what's the wifi password?"  
Top score: 0.32 — below the 0.4 threshold  
Best match: kb-07 ("reset your password")

This shows that a similarity threshold can act as a safety net.
If the top score is low, we can return "I don't have an answer for this"
instead of surfacing a misleading result.